# SBE26 Data Processing

Processing path used:

proc_1 IMOS NetCDF -> manual QC flagging -> proc_2 IMOS NetCDF

### Setup

Imports

In [ ]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

Import local tools

In [ ]:
TOOLS_DIR = Path.cwd().resolve().parent
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from tools.imos_nc_converter import imos_converter as imos_converter_module
importlib.reload(imos_converter_module)
IMOSNetCDFConverter_SBE26 = imos_converter_module.IMOSNetCDFConverter_SBE26

from tools import database_lookup as database_lookup_module
importlib.reload(database_lookup_module)
get_instrument_context = database_lookup_module.get_instrument_context
update_metadata_file_fields = database_lookup_module.update_metadata_file_fields

from tools.helpers import apply_qc_flag_windows, plot_data_by_qc, write_manual_qc_flags_txt

Definitions

In [ ]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash")

In [ ]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 259    # update per deployment

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    print_details=True,
)

In [ ]:
converter = IMOSNetCDFConverter_SBE26(input_folder="", input_file="", output_dir="")

### QC

Read proc_1 dataset

In [ ]:
def resolve_stage_dir(path_value):
    stage_dir = Path(str(path_value)).expanduser()
    if not stage_dir.is_absolute():
        stage_dir = (Path.cwd() / stage_dir).resolve()
    return stage_dir

def find_proc_1_file(stage_dir):
    configured_name = _row.get("proc_1_file", None)
    if pd.notna(configured_name) and str(configured_name).strip():
        configured_path = stage_dir / str(configured_name).strip()
        if configured_path.exists():
            return configured_path

    pattern = f"{_row['location']}_*_SBE26_{int(_row['inst_id'])}_*.nc"
    candidates = sorted(stage_dir.glob(pattern))
    if not candidates:
        candidates = sorted(stage_dir.glob("*.nc"))
    if not candidates:
        raise FileNotFoundError(f"No proc_1 NetCDF files found in {stage_dir}")
    return candidates[-1]

proc_1_dir = resolve_stage_dir(_row["proc_1_path"])
proc_1_path = find_proc_1_file(proc_1_dir)
with xr.open_dataset(proc_1_path) as opened_ds:
    ds_proc1 = opened_ds.load()

print(f"Using proc_1 input: {proc_1_path}")

In [ ]:
time_coverage_start = ds_proc1.attrs.get("time_coverage_start", _row.get("deploy_date", None))
time_coverage_end = ds_proc1.attrs.get("time_coverage_end", _row.get("recovery_date", None))
deploy_start = _row.get("deploy_date", time_coverage_start)
deploy_end = _row.get("recovery_date", time_coverage_end)
inst_channels = ds_proc1.attrs.get("mooring_channels", _row.get("mooring_channels", "PT"))
converter_depth = float(_row.get("nominal_depth", 0.0))

print(f"start: {time_coverage_start}, end: {time_coverage_end}")

### Proc_2

Manual QC - flagging

In [ ]:
ds_proc2 = ds_proc1.copy()


In [ ]:
PLOT_VARS = None  # set None for defaults, to specify use ["PRES", "TEMP"]
fig = plot_data_by_qc(
    ds_proc2,
    variables=PLOT_VARS,
    # flags_to_plot=[1],      # optional
    # y_zoom_to_good=True,    # optional
)
fig.show()

In [ ]:

manual_qc_flags = [
    # {"qc_vars": ["PRES_quality_control"], "flag": 3, "start": "YYYY-mm-dd HH:MM:SS", "end": "YYYY-mm-dd HH:MM:SS"},
]

ds_proc2 = apply_qc_flag_windows(ds_proc2, manual_qc_flags, time_name="TIME")

Save QC to proc_2 dataset and output

In [ ]:
proc_2_source_path = str((Path.cwd() / "proc_2_source_sbe26.nc").resolve())
ds_proc2.to_netcdf(proc_2_source_path)

proc_2_out = converter.process(
    input_nc_path=proc_2_source_path,
    longitude=float(_row["longitude"]),
    latitude=float(_row["latitude"]),
    depth=converter_depth,
    inst_channels=str(inst_channels),
    start_of_good_data=time_coverage_start,
    time_deployment_start=deploy_start,
    time_deployment_end=deploy_end,
    site_code=str(_row["location"]),
    version=str(_row.get("version", "1")),
    instrument=str(_row["inst_type"]),
    inst_id=str(int(_row["inst_id"])),
    location=str(_row["location"]),
    output_name_mode="internal",
    output_stage="proc_2",
    metadata_row=_row,
    metadata_mode="fill_missing",
    deployment_id=_row.get("deployment_id", ""),
    mooring_channels=_row.get("mooring_channels", ""),
    nominal_inst_depth=_row.get("nominal_inst_depth", ""),
    deploy_date=_row.get("deploy_date", ""),
    recovery_date=_row.get("recovery_date", ""),
    time_coverage_start=time_coverage_start,
    time_coverage_end=time_coverage_end,
)

manual_qc_log_path = write_manual_qc_flags_txt(proc_2_out, manual_qc_flags)

print(f"proc_2 output: {proc_2_out}")
print(f"manual QC flags log: {manual_qc_log_path}")
Path(proc_2_source_path).unlink(missing_ok=True)

Write proc_2 filename to metadata table

In [ ]:
proc_2_name = Path(proc_2_out).name
_row = update_metadata_file_fields(
    inst_deploy_id,
    {"proc_2_file": proc_2_name},
    output_paths={"proc_2_file": proc_2_out},
    working_dir=Path.cwd(),
)
print(f"Updated proc_2_file: {_row['proc_2_file']}")